In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
SPARQL_ENDPOINT = os.getenv('SPARQL_ENDPOINT')
LLM_ENDPOINT = os.getenv('LLM_ENDPOINT')

In [2]:
from openai import OpenAI

def generate_response(user_prompt: str, messages: list):
    """Calls the LLM with history.
        input (str): Input of the LLM.
        messages (list): History of messages.
    """
    try:
        user_message = {"role": "user", "content": user_prompt}
        messages.append(user_message)

        client = OpenAI(base_url=LLM_ENDPOINT, api_key="lm-studio")

        response = client.chat.completions.create(
            model="lmstudio-community/Meta-Llama-3.1-8B-Instruct-GGUF",
            messages=messages,
            temperature=1,
        )

        new_message = {"role": "assistant", "content": response.choices[0].message.content}
        messages.append(new_message)
    
    except Exception as e:
        print(f"An error occured when invoking the LLM: {e}")

    return response, messages


def invoke_llm(system_prompt: str, user_prompt: str):
    """Calls the LLM without history.
    LLM function call to be passed to strictjson.strict_json(). It is advised not to change the parameters.
    Refer to https://github.com/tanchongmin/strictjson/blob/main/strictjson/base.py#L319

    Args:
        system_prompt (str): System prompt.
        user_prompt (str): User prompt.

    Returns:
        str: LLM response string.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]   

    client = OpenAI(base_url=LLM_ENDPOINT, api_key="lm-studio")

    response = client.chat.completions.create(
        model="lmstudio-community/Meta-Llama-3.1-8B-Instruct-GGUF",
        messages=messages,
        temperature=1,
    )

    return response.choices[0].message.content


########################################################################
## Entity recognition ##################################################
########################################################################
ER_SYSTEM_PROMPT = """
You are an expert in the gastronomy domain with extensive knowledge of various pizzas. Your task is to 
accurately identify and extract the names of entities from the given text. You should focus on recognizing 
specific terminology and contextually relevant phrases that pertain to objects such as food and beverages 
within the gastronomy sector. Your output should be in JSON format, placing the identified terms into the key
"entities". For instance {'entities': ['LaReine', 'MeatyPizza', 'Mushroom']} The output should be strictly 
the JSON object without any additional commentary or explanation.
"""

ER_USER_PROMPT = """
I want you to identify the entities of the following input.

Input: {input}
"""

########################################################################
## SPARQL generation ###################################################
########################################################################
CONTEXT_RETRIEVAL_SYSTEM_PROMPT = """
You are a helpful assistant. I want you to answer generate a SPARQL query considering the question given by the user and the following pandas DataFrame containing RDF triples:
    {context}
-Use only classes and properties defined in the RDF graph, for this is important to use the same URIs for the properties and classes as defined in the original graph; 
-Include all the prefixes used in the SPARQL query; 
-Declare non-essential properties to the question as OPTIONAL if needed; 
-DO NOT use specific resources in the query; Declare filters on strings (like labels and names) as filter operations over the REGEX function using the case-insensitive flag.

Your output should be in JSON format, categorizing the identified terms into "sparql_query" and "explanation". The output should be strictly the JSON object without any additional commentary or explanation.
"""

CONTEXT_RETRIEVAL_USER_PROMPT = """
Question: {question}
"""

########################################################################
## QA prompt template ##################################################
########################################################################
QA_SYSTEM_PROMPT = """
You are a helpful assistant. I want you to answer the given user question considering the context given in the
following RDF triples. Explain your reasoning and cite the relevant triples
Triples: {context}
"""

# TODO: Enhance the prompt.
QA_USER_PROMPT = """
Question: {question}
"""

In [3]:
# utils.py

import pandas as pd

def replace_urls_with_prefixes(text, prefix_dict):
    """Replace URLs with prefixes in a given string.

    Args:
        text (str): String containing SPARQL query or text.
        prefix_dict (dict): Mapping of prefixes to URLs.
    """
    for prefix, url in prefix_dict.items():
        text = text.replace(url, prefix)
    return text


def replace_prefixes_with_urls(text, prefix_dict):
    """Replace prefixes with URLs in a given string.

    Args:
        text (str): String containing SPARQL query or text.
        prefix_dict (dict): Mapping of prefixes to URLs.
    """
    for prefix, url in prefix_dict.items():
        text = text.replace(prefix, url)
    return text


def parse_sparql_output(query_output: dict):
    """Parse query JSON output to dataframe.

    Args:
        query_output (dict): The output from a SPARQL query in JSON format

    Returns:
        pd.DataFrame: the parsed results as a pandas DataFrame
    """
    columns = query_output['head']['vars']
    rows = []
    for result in query_output['results']['bindings']:
        row = {}
        for col in columns:
            row[col] = result[col]['value'] if col in result else None
        rows.append(row)
    df = pd.DataFrame(rows, columns=columns)
    return df


def apply_score_threshold():
    pass


In [4]:
# sparql.py

from manon_chat_interface.utils import utils
from urllib.error import URLError
from rdflib import Graph, BNode
from SPARQLWrapper import SPARQLWrapper, JSON


def execute_sparql(url: str, query: str):
    """    
    Executes a SPARQL query against a specified endpoint.

    This function sends a SPARQL query to the given endpoint URL and returns the results in JSON format.

    Args:
        url (str): The URL of the SPARQL query endpoint.
        query (str): The SPARQL query to be executed.

    Returns:
        dict: The results of the SPARQL query in JSON format.
    """
    try:
      wrapper = SPARQLWrapper(url)
      wrapper.setQuery(query)
      wrapper.setReturnFormat(JSON)
      results = wrapper.query().convert()

    except URLError as e:
        print(f"\033[91mERROR. Server may not be running.\033[0m Error message: {e}")
        
    return results


def execute_parse_sparql(url, queries):
    """Helper function for retrieve_context().

    Args:
        url (_type_): _description_
        queries (_type_): _description_

    Returns:
        _type_: _description_
    """
    results = []
    for index, query in enumerate(queries):
        result = execute_sparql(url=url, query=PREFIXES+query)
        table = parse_sparql_output(result).to_string()
        results.append(f"Triples: {index}\n{table}")
    subgraph = "\n".join(results)
    subgraph = replace_urls_with_prefixes(subgraph, prefix_dict)
    return subgraph


def load_rdf_triples(file_path: str, format: str = 'ttl'):
    
  # Create a Graph object 
  g = Graph()
  g.parse(file_path, format=format)

  # Retrieve triples whose subject and object are not blank nodes
  triples_string = ""
  for subj, pred, obj in g:
      if not isinstance(subj, BNode) and not isinstance(obj, BNode):
          triples_string += f"{subj} {pred} {obj} .\n"

  replaced_triples_string = replace_urls_with_prefixes(triples_string, prefix_dict)

  return replaced_triples_string


prefix_dict = {
    ":": "http://www.co-ode.org/ontologies/pizza#",
    "dc:": "http://purl.org/dc/elements/1.1/",
    "owl:": "http://www.w3.org/2002/07/owl#",
    "rdf:": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "xml:": "http://www.w3.org/XML/1998/namespace",
    "xsd:": "http://www.w3.org/2001/XMLSchema#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "skos:": "http://www.w3.org/2004/02/skos/core#",
    "pizza:": "http://www.co-ode.org/ontologies/pizza/pizza.owl#",
    "terms:": "http://purl.org/dc/terms/",
    "base:": "http://www.co-ode.org/ontologies/pizza#"
}

PREFIXES = """
PREFIX : <http://www.co-ode.org/ontologies/pizza#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xml: <http://www.w3.org/XML/1998/namespace>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX pizza: <http://www.co-ode.org/ontologies/pizza/pizza.owl#>
PREFIX terms: <http://purl.org/dc/terms/>
BASE <http://www.co-ode.org/ontologies/pizza#>
"""

########################################################################
## Embed entities ######################################################
########################################################################
EXTRACTION_QUERY = """
SELECT DISTINCT ?individual
WHERE {{
{{
 SELECT ?individual
 WHERE {{
   ?individual rdfs:subClassOf* {pizza_class} .
 }}
}}
UNION
{{
 SELECT DISTINCT ?individual
 WHERE {{
   ?individual rdf:type owl:Class ;
           owl:equivalentClass ?equivClass .

   ?equivClass owl:intersectionOf ?list .
   ?list rdf:rest*/rdf:first {pizza_class} .
 }}
}}
}}
"""

########################################################################
## n-Hop ###############################################################
########################################################################
ONE_HOP = """
SELECT ?subject ?predicate ?object WHERE {{
  {{
    ?s ?p <{IRI}> .
    BIND(?s AS ?subject)
    BIND(?p AS ?predicate)
    BIND(<{IRI}> AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    <{IRI}> ?p ?o .
    BIND(<{IRI}> AS ?subject)
    BIND(?p AS ?predicate)
    BIND(?o AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
}}
"""

TWO_HOP = """
SELECT ?subject ?predicate ?object WHERE {{
  {{
    ?s ?p <{IRI}> .
    BIND(?s AS ?subject)
    BIND(?p AS ?predicate)
    BIND(<{IRI}> AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    ?s ?p <{IRI}> .
    ?s1 ?p1 ?s .
    BIND(?s1 AS ?subject)
    BIND(?p1 AS ?predicate)
    BIND(?s AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    <{IRI}> ?p ?o .
    BIND(<{IRI}> AS ?subject)
    BIND(?p AS ?predicate)
    BIND(?o AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    <{IRI}> ?p ?o .
    ?o ?p1 ?o1 .
    BIND(?o AS ?subject)
    BIND(?p1 AS ?predicate)
    BIND(?o1 AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
}}
"""

THREE_HOP = """
SELECT ?subject ?predicate ?object WHERE {{
  {{
    ?s ?p <{IRI}> .
    BIND(?s AS ?subject)
    BIND(?p AS ?predicate)
    BIND(<{IRI}> AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    ?s ?p <{IRI}> .
    ?s1 ?p1 ?s .
    BIND(?s1 AS ?subject)
    BIND(?p1 AS ?predicate)
    BIND(?s AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    ?s ?p <{IRI}> .
    ?s1 ?p1 ?s .
    ?s2 ?p2 ?s1 .
    BIND(?s2 AS ?subject)
    BIND(?p2 AS ?predicate)
    BIND(?s1 AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    <{IRI}> ?p ?o .
    BIND(<{IRI}> AS ?subject)
    BIND(?p AS ?predicate)
    BIND(?o AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    <{IRI}> ?p ?o .
    ?o ?p1 ?o1 .
    BIND(?o AS ?subject)
    BIND(?p1 AS ?predicate)
    BIND(?o1 AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
  UNION
  {{
    <{IRI}> ?p ?o .
    ?o ?p1 ?o1 .
    ?o1 ?p2 ?o2 .
    BIND(?o1 AS ?subject)
    BIND(?p2 AS ?predicate)
    BIND(?o2 AS ?object)
    FILTER (!isBlank(?subject) && !isBlank(?object))
  }}
}}
"""


ModuleNotFoundError: No module named 'manon_chat_interface.utils'

In [5]:
# orchestration.py
from strictjson import strict_json

import chromadb


def entity_recognition(input: str) -> dict:
    """
    Performs entity recognition on the provided input string using a large language model (LLM).

    This function identifies and categorizes entities within the user input.

    Multiple 

    Args:
        input (str): The input string to be analyzed by the LLM.

    Returns:
        dict: A dictionary with key 'entities', containing the identified entities within the question.

    Example:
        Input: "Can Creality Ender manufacture flange?"
        Output: {'entities': ['Creality Ender', 'flange']}

    """
    response = strict_json(
        system_prompt=ER_SYSTEM_PROMPT,
        user_prompt=ER_USER_PROMPT.format(input=input),
        output_format={"entities": "Array of entities"},
        llm=invoke_llm
    )

    return response



def get_triples(
        question: str, # Original question 
        mode: str = "default",
        path_to_vectorstore: str = "./manon_chat_interface/data/vectorstores", 
        path_to_graph: str = None, # Path to RDF graph
        format: str = None, # Format of RDF graph
):
    """Get triples to be given as context to the QA model.

    Args:
        question (str): Question.
        path_to_vectorstore (str, optional): Path to the vectorstore.
        path_to_graph (str, optional): Path to the file containing the RDF graph.

    Raises:
        ValueError: If required params not given.
        Exception: If mode is invalid.

    Returns:
        str: Text containing RDF triples.
    """

    # Get chroma. Adjust client name if necessary.
    entities_client = chromadb.PersistentClient(path=f"{path_to_vectorstore}/pizza_entities")
    questions_client = chromadb.PersistentClient(path=f"{path_to_vectorstore}/pizza_questions")

    pizza_collection = entities_client.get_or_create_collection("pizza_collection")
    question_collection = questions_client.get_or_create_collection("pizza_questions")

    # Map question
    question_emb = question_collection.query(
        query_texts=[question], 
        n_results=1
    )
    mapped_question = question_emb["documents"][0][0]

    # Map entities
    entities = entity_recognition(question)["entities"] # List of entities
    entities_emb = pizza_collection.query(
        query_texts=entities, 
        n_results=1
    )
    mapped_entities = [metadata[0]['IRI'] for metadata in entities_emb['metadatas']]

    if mode == "default":
        if path_to_graph is None or format is None:
            raise ValueError("In 'default' mode, 'path_to_graph' and 'format' cannot be None.")

        triples = load_rdf_triples(file_path=path_to_graph, format=format)
        
        return triples
    
    elif mode == "n_hop":
        # TODO: Add cases for 1, 2, 3 hops
        triples = ""
        
        for iri in mapped_entities:
            query = PREFIXES+TWO_HOP.format(IRI=iri)
            query_result = execute_parse_sparql(
                url=SPARQL_ENDPOINT, 
                queries=[query]
            )
            triples += query_result

        return triples

    elif mode == "generated":
        # LLM Generation
        if path_to_graph is None or format is None:
            raise ValueError("In 'default' mode, 'path_to_graph' and 'format' cannot be None.")
        
        triples = load_rdf_triples(file_path=path_to_graph, format=format)

        query = strict_json(
            system_prompt=CONTEXT_RETRIEVAL_SYSTEM_PROMPT.format(
                context=triples
            ),
            user_prompt=CONTEXT_RETRIEVAL_USER_PROMPT.format(
                question=question
            ),
            output_format={
                "sparql_query": "Executable sparql query string.", 
                "explanation": "Explanation of what the sparql query does."
            },
            llm=invoke_llm
        )
        # try:
        #     response = execute_parse_sparql(url, [query])
        
        # except Exception as e:
        #     print(f"The following error occured when executing generated query:/n{e}")
        
        return query
    
    else:
        raise Exception("Given mode is not valid. Choose between 'default', 'n-hop', or 'generated'")




### main.py

In [7]:
#PATH!

path_to_entities = "../../manon_chat_interface/data/vectorstores/pizza_entities"
path_to_questions = "../../manon_chat_interface/data/vectorstores/pizza_questions"

entities_client = chromadb.PersistentClient(path=path_to_entities)
questions_client = chromadb.PersistentClient(path=path_to_questions)


# Get question
original_question = "Can I make an american pizza with mozzarella topping?"
question_collection = questions_client.get_or_create_collection("pizza_questions")
question_emb = question_collection.query(
    query_texts=[original_question], n_results=1
)


In [8]:
mapped_question = question_emb["documents"][0][0] # Context
mapped_question

'Do I need this particular topping for my pizza?'

In [20]:
# Get entities' IRIs
pizza_collection = entities_client.get_or_create_collection("pizza_collection")
# topping_collection = entities_client.get_or_create_collection("topping_collection")

entities = entity_recognition(original_question) # List of entities
entities = entities["entities"]
# recognized_pizza = entity["pizza"]
# recognized_topping = entity["topping"]

entities_emb = pizza_collection.query(
    query_texts=['American Pizza', 'Mozzarella', 'Americ', 'American'],
    n_results=1
)
# TODO: Apply cosine threshold and entity not found handling.
# Get the documents where the corresponding distance is below the threshold
threshold = 0.5
# Get the documents and IRIs where the corresponding distance is below the threshold
filtered_results = [
    (doc[0], meta[0]['IRI']) 
    for doc, dist, meta in zip(entities_emb['documents'], entities_emb['distances'], entities_emb['metadatas']) 
    if dist[0] < threshold
]

# Separate documents and IRIs into different lists
filtered_documents = [result[0] for result in filtered_results]
filtered_iris = [result[1] for result in filtered_results]

entities_iris = [metadata[0]['IRI'] for metadata in entities_emb['metadatas']]

In [22]:
entities_emb

{'ids': [['af38530c-db3a-5a1d-95cf-f1eeb0dd5cfc'],
  ['ed1ce158-71f3-59ba-83db-9ddcb71921cf'],
  ['5a6baa56-bc1f-58ab-9fa6-f4b2f8659164'],
  ['5a6baa56-bc1f-58ab-9fa6-f4b2f8659164']],
 'distances': [[0.2231055301387438],
  [0.15612258989051442],
  [0.6455360954342937],
  [0.0]],
 'metadatas': [[{'IRI': 'http://www.co-ode.org/ontologies/pizza/pizza.owl#Pizza'}],
  [{'IRI': 'http://www.co-ode.org/ontologies/pizza/pizza.owl#MozzarellaTopping'}],
  [{'IRI': 'http://www.co-ode.org/ontologies/pizza/pizza.owl#American'}],
  [{'IRI': 'http://www.co-ode.org/ontologies/pizza/pizza.owl#American'}]],
 'embeddings': None,
 'documents': [['Pizza'], ['MozzarellaTopping'], ['American'], ['American']],
 'uris': None,
 'data': None}

In [23]:
filtered_documents

['Pizza', 'MozzarellaTopping', 'American']

In [24]:
filtered_iris

['http://www.co-ode.org/ontologies/pizza/pizza.owl#Pizza',
 'http://www.co-ode.org/ontologies/pizza/pizza.owl#MozzarellaTopping',
 'http://www.co-ode.org/ontologies/pizza/pizza.owl#American']